In [1]:
import sys
import os
import logging
from collections import Counter
# pyrefly: ignore [missing-import]
from datasets import load_dataset, DatasetDict, concatenate_datasets
from typing import List
sys.path.append(os.path.abspath('..'))
os.chdir('..')
from transformers import AutoTokenizer

from src.utils.config import load_config
from src.data.preprocessor import MultilingualPreprocessor

In [15]:
data_config = load_config("configs/data_config.yaml")
model_config = load_config("configs/model_config.yaml")
cache_dir = "./data/raw"
max_samples = data_config['max_samples']
val_samples = data_config['val_samples']
test_samples = data_config['test_samples']

In [4]:
print("\n2. Bắt đầu tải OPUS-100 với các tham số:")
print(f"- Các cặp ngôn ngữ: {data_config['lang_pairs']}")
print(f"- Max samples (train): {data_config['max_samples']}")
print(f"- Val samples: {data_config['val_samples']}")
print(f"- Test samples: {data_config['test_samples']}")


2. Bắt đầu tải OPUS-100 với các tham số:
- Các cặp ngôn ngữ: ['en-vi', 'en-fr', 'de-en']
- Max samples (train): 500
- Val samples: 10
- Test samples: 20


In [ ]:
lang_pair = data_config['lang_pairs']
all_train, all_val, all_test = [], [], []
for pair in lang_pair:
    src, tgt = pair.split('-')

    langs = sorted([src, tgt])
    config_name = f"{langs[0]}-{langs[1]}"
    ds = load_dataset("Helsinki-NLP/opus-100", config_name, cache_dir=cache_dir)

    train = ds["train"].select(range(min(len(ds["train"]), max_samples)))
    val = ds["validation"] if "validation" in ds else ds["test"].select(range(min(len(ds["test"]), val_samples)))
    test = ds["test"].select(range(min(len(ds["test"]), test_samples)))

    train = train.map(lambda x: {"pair": pair, "src": x["translation"][src], "tgt": x["translation"][tgt]}, remove_columns=["translation"])
    val = val.map(lambda x: {"pair": pair, "src": x["translation"][src], "tgt": x["translation"][tgt]}, remove_columns=["translation"])
    test = test.map(lambda x: {"pair": pair, "src": x["translation"][src], "tgt": x["translation"][tgt]}, remove_columns=["translation"])

    all_train.append(train)
    all_val.append(val)
    all_test.append(test)


In [31]:
dataset = DatasetDict({
    "train": concatenate_datasets(all_train).shuffle(seed=42),
    "validation": concatenate_datasets(all_val),
    "test": concatenate_datasets(all_test)
})

# In phân phối dữ liệu
pair_counts = Counter(dataset["train"]["pair"])

In [ ]:
model_name = model_config["model_name"]
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

In [30]:
preprocessor = MultilingualPreprocessor(tokenizer, data_config['max_length'])

In [ ]:
tokenized_dataset = dataset.map(
    preprocessor.preprocess_function, 
    batched=True,       
    batch_size=100
)

In [2]:
import os
import sys
from collections import Counter

from src.utils.logger import setup_logger
from src.utils.seed import set_seed
from src.utils.config import load_config
from src.data.loader import load_multilingual_dataset
from src.data.preprocessor import MultilingualPreprocessor
from src.data.collator import custom_collate_fn
from src.model.tokenizer import load_mbart_tokenizer
from src.model.builder import load_mbart_model
from src.training.arguments import get_training_args
from src.training.callbacks import get_callbacks
from src.training.trainer import BalancedLossSeq2SeqTrainer
from src.evaluation.metrics import get_compute_metrics

# 1. Môi trường cơ sở
logger = setup_logger("train_script")
set_seed(42)
logger.info("Bắt đầu kịch bản huấn luyện NMT Đa Ngôn Ngữ với Balanced Loss!")

# 2. Tải cấu hình
data_config = load_config("configs/data_config.yaml")
train_config = load_config("configs/train_config.yaml")
model_config = load_config("configs/model_config.yaml")

2026-06-11 13:11:04 - train_script - INFO - Bắt đầu kịch bản huấn luyện NMT Đa Ngôn Ngữ với Balanced Loss!


In [3]:
logger.info("Đang tải dữ liệu OPUS-100...")
dataset = load_multilingual_dataset(
    lang_pairs=data_config["lang_pairs"],
    max_samples=data_config["max_samples"],
    val_samples=data_config["val_samples"],
    test_samples=data_config["test_samples"]
)

2026-06-11 13:11:06 - train_script - INFO - Đang tải dữ liệu OPUS-100...


In [4]:
# Tính tần suất ngôn ngữ từ tập Train để chia trọng số phạt
pair_counts = dict(Counter(dataset["train"]["pair"]))
logger.info(f"Tần suất cặp ngôn ngữ (phục vụ Balanced Loss): {pair_counts}")

# 4. Tải Model & Tokenizer
tokenizer = load_mbart_tokenizer(model_config["model_name"], "./models")
model = load_mbart_model(model_config["model_name"], './models')

# 5. Tiền xử lý dữ liệu (Tokenization)
logger.info("Tiến hành Tokenize toàn bộ tập dữ liệu...")
preprocessor = MultilingualPreprocessor(tokenizer, max_length=data_config["max_length"])

tokenized_datasets = dataset.map(
    preprocessor.preprocess_function,
    batched=True,
    batch_size=1000,
    remove_columns=["src", "tgt"] # Xóa cột văn bản thô, nhưng GIỮ LẠI cột 'pair'
)

2026-06-11 13:11:41 - train_script - INFO - Tần suất cặp ngôn ngữ (phục vụ Balanced Loss): {'en-fr': 500, 'en-vi': 500, 'de-en': 500}


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

2026-06-11 13:11:44 - train_script - INFO - Tiến hành Tokenize toàn bộ tập dữ liệu...


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [5]:
# 6. Cấu hình Training
training_args = get_training_args(train_config, output_dir="./saved_models/mbart50-balanced")
callbacks = get_callbacks()

# 7. Khởi tạo BalancedLossSeq2SeqTrainer
logger.info("Gắn kết Model, Data, Loss vào Trainer...")
trainer = BalancedLossSeq2SeqTrainer(
    pair_counts=pair_counts,
    smoothing_factor=0.5,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=custom_collate_fn, 
    compute_metrics=get_compute_metrics(tokenizer),
    callbacks=callbacks
)

# 8. Bắt đầu Vòng lặp Huấn luyện (Training Loop)
logger.info("🚀 Bắt đầu HUẤN LUYỆN!")
trainer.train()

# 9. Lưu trữ sau khi chạy xong
logger.info("Đang lưu mô hình hoàn chỉnh...")
trainer.save_model("./saved_models/mbart50-balanced-final")
tokenizer.save_pretrained("./saved_models/mbart50-balanced-final")
logger.info("Tuyệt vời! Đã hoàn tất huấn luyện!")



[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


2026-06-11 13:11:50 - train_script - INFO - Gắn kết Model, Data, Loss vào Trainer...
2026-06-11 13:11:59 - train_script - INFO - 🚀 Bắt đầu HUẤN LUYỆN!


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.69 GiB is allocated by PyTorch, and 109.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)